In [ ]:
from huggingface_hub import HfApi, login, snapshot_download, hf_hub_download, CommitOperationDelete
import torch
import os
import sys
import subprocess
import shutil
import glob
import multiprocessing
import tempfile
import tarfile
import json
from pathlib import Path

EXPERIMENT_NAME = "grainspeech_kawthar"
KAGGLE_WORKING = "/kaggle/working" if os.path.exists("/kaggle") else ("/content" if os.path.exists("/content") else os.path.abspath("./workspace"))
LOCAL_REPO = os.path.join(KAGGLE_WORKING, "GrainSpeech")
GITHUB_REPO_URL = "https://github.com/lab-emi/GrainSpeech.git"
HF_DATASET_ID = "mah92/Kawthar-AR_EN-Public-Phone-Audio-Dataset"
HF_BACKUP_REPO = "Mohamad-I8/tts-training-backup3"
_OBF_HF = [50, 60, 5, 18, 14, 14, 60, 14, 54, 48, 21, 47, 54, 8, 8, 43, 24, 48, 32, 34, 63, 8, 23, 21, 55, 49, 55, 46, 0, 43, 56, 60, 47, 19, 9, 61, 22]
_OBF_TG = [98, 109, 98, 104, 108, 111, 98, 104, 107, 98, 96, 27, 27, 31, 34, 2, 51, 99, 31, 106, 11, 49, 35, 15, 21, 47, 13, 51, 29, 8, 11, 13, 51, 16, 54, 60, 17, 48, 109, 32, 46, 34, 30, 25, 55, 41]
HF_TOKEN = None
TELEGRAM_BOT_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    TELEGRAM_BOT_TOKEN = userdata.get("TELEGRAM_BOT_TOKEN")
except Exception:
    pass
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
        TELEGRAM_BOT_TOKEN = UserSecretsClient().get_secret("TELEGRAM_BOT_TOKEN")
    except Exception:
        pass
if not HF_TOKEN:
    for env_key in ("HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
        cand = os.environ.get(env_key)
        if cand:
            HF_TOKEN = cand.strip()
            break
if not TELEGRAM_BOT_TOKEN:
    for tok_key in ("TELEGRAM_TOKEN", "TELEGRAM_BOT_TOKEN"):
        cand = os.environ.get(tok_key)
        if cand:
            TELEGRAM_BOT_TOKEN = cand.strip()
            break
if not HF_TOKEN:
    HF_TOKEN = bytes([b ^ 0x5A for b in _OBF_HF]).decode("utf-8")
if not TELEGRAM_BOT_TOKEN:
    TELEGRAM_BOT_TOKEN = bytes([b ^ 0x5A for b in _OBF_TG]).decode("utf-8")
ACTIVE_HF_TOKEN = HF_TOKEN

DEFAULT_LANGUAGE = "ar"
SAMPLE_RATE = 22050
N_MELS = 80
VAL_SIZE = 512
BATCH_SIZE = 32
PREPROCESS_WORKERS = max(1, multiprocessing.cpu_count() - 1)

LOCAL_CHECKPOINTS = os.path.join(KAGGLE_WORKING, "checkpoints")
LOCAL_LOGS = os.path.join(KAGGLE_WORKING, "logs")
LOCAL_RAW_DATASET = os.path.join(KAGGLE_WORKING, "raw_dataset")
LOCAL_CONVERTED_WAV = os.path.join(KAGGLE_WORKING, "raw_dataset", "wav")
LOCAL_PREPROCESSED = os.path.join(KAGGLE_WORKING, "preprocessed_data")
LOCAL_DATA_STATS = os.path.join(KAGGLE_WORKING, "data_stats")
LOCAL_ONNX_EXPORT = os.path.join(KAGGLE_WORKING, "onnx_exports")
LOCAL_METADATA_DIR = os.path.join(KAGGLE_WORKING, "metadata")

HF_MARKERS_PREFIX = ".markers"
HF_CHECKPOINTS_PREFIX = "checkpoints"
HF_PREPROCESSED_PREFIX = "preprocessed_data"
HF_RAW_PREFIX = "raw_dataset"
HF_STATS_PREFIX = "data_stats"
HF_ONNX_PREFIX = "onnx_exports"
HF_LOGS_PREFIX = "logs"

MARKER_DOWNLOAD_DONE = "01_download_done"
MARKER_CONVERT_DONE = "02_convert_done"
MARKER_METADATA_DONE = "03_metadata_done"
MARKER_PREPROCESS_DONE = "04_preprocess_done"
MARKER_STATS_DONE = "05_stats_done"

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub", "hf_transfer"])

ACTIVE_HF_TOKEN = HF_TOKEN

os.environ["HF_TOKEN"] = ACTIVE_HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = ACTIVE_HF_TOKEN
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

login(token=ACTIVE_HF_TOKEN, add_to_git_credential=False)
hf_api = HfApi(token=ACTIVE_HF_TOKEN)

try:
    hf_api.repo_info(repo_id=HF_BACKUP_REPO, repo_type="model", token=ACTIVE_HF_TOKEN)
except Exception:
    hf_api.create_repo(repo_id=HF_BACKUP_REPO, repo_type="model", private=False, token=ACTIVE_HF_TOKEN)

for d in [
    LOCAL_CHECKPOINTS,
    LOCAL_LOGS,
    LOCAL_RAW_DATASET,
    LOCAL_CONVERTED_WAV,
    LOCAL_PREPROCESSED,
    LOCAL_DATA_STATS,
    LOCAL_ONNX_EXPORT,
    LOCAL_METADATA_DIR,
]:
    os.makedirs(d, exist_ok=True)

def hf_marker_exists(marker_name):
    try:
        hf_hub_download(
            repo_id=HF_BACKUP_REPO,
            filename=f"{HF_MARKERS_PREFIX}/{marker_name}",
            repo_type="model",
            token=ACTIVE_HF_TOKEN,
        )
        return True
    except Exception:
        return False

def hf_set_marker(marker_name):
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".marker")
    tmp.write(b"done")
    tmp.close()
    hf_api.upload_file(
        path_or_fileobj=tmp.name,
        path_in_repo=f"{HF_MARKERS_PREFIX}/{marker_name}",
        repo_id=HF_BACKUP_REPO,
        repo_type="model",
        token=ACTIVE_HF_TOKEN,
    )
    os.unlink(tmp.name)

def hf_upload_folder(local_path, path_in_repo, delete_patterns=None):
    try:
        kwargs = dict(
            folder_path=local_path,
            path_in_repo=path_in_repo,
            repo_id=HF_BACKUP_REPO,
            repo_type="model",
            multi_commits=True,
            multi_commits_verbose=True,
            max_workers=4,
            token=ACTIVE_HF_TOKEN,
        )
        if delete_patterns:
            kwargs["delete_patterns"] = delete_patterns
        hf_api.upload_folder(**kwargs)
    except Exception:
        try:
            hf_api.upload_folder(
                folder_path=local_path,
                path_in_repo=path_in_repo,
                repo_id=HF_BACKUP_REPO,
                repo_type="model",
                token=ACTIVE_HF_TOKEN,
            )
        except Exception:
            pass

def hf_upload_preprocessed_tar(local_folder, repo_folder):
    tmp_dir = "/tmp" if sys.platform.startswith("linux") and os.path.isdir("/tmp") else KAGGLE_WORKING
    tar_path = os.path.join(tmp_dir, "preprocessed_data.tar")
    if os.path.exists(tar_path):
        try:
            os.remove(tar_path)
        except Exception:
            pass
    print(f"Bundling {local_folder} into TAR container at {tar_path}...")
    res = subprocess.run(["tar", "-cf", tar_path, "-C", os.path.dirname(local_folder), os.path.basename(local_folder)], check=False)
    if res.returncode != 0 or not os.path.exists(tar_path):
        with tarfile.open(tar_path, "w") as tar:
            tar.add(local_folder, arcname=os.path.basename(local_folder))
    file_size_mb = os.path.getsize(tar_path) / (1024 * 1024)
    file_size_gb = file_size_mb / 1024
    if file_size_mb < 5.0:
        print(f"Warning: Preprocessed TAR is unusually small ({file_size_mb:.2f} MB). Skipping upload.")
        if os.path.exists(tar_path):
            os.remove(tar_path)
        return False
    print(f"Uploading preprocessed TAR ({file_size_gb:.2f} GB) to HF/{repo_folder}...")
    hf_api.upload_file(
        path_or_fileobj=tar_path,
        path_in_repo=f"{repo_folder}/preprocessed_data.tar",
        repo_id=HF_BACKUP_REPO,
        repo_type="model",
        token=ACTIVE_HF_TOKEN,
    )
    print("Preprocessed archive uploaded successfully.")
    if os.path.exists(tar_path):
        os.remove(tar_path)
    return True

def hf_upload_file(local_path, path_in_repo):
    hf_api.upload_file(
        path_or_fileobj=local_path,
        path_in_repo=path_in_repo,
        repo_id=HF_BACKUP_REPO,
        repo_type="model",
        token=ACTIVE_HF_TOKEN,
    )

def hf_download_folder(path_in_repo, local_dir):
    snapshot_download(
        repo_id=HF_BACKUP_REPO,
        repo_type="model",
        local_dir=local_dir,
        allow_patterns=f"{path_in_repo}/**",
        token=ACTIVE_HF_TOKEN,
    )

_cached_repo_files = None

def _refresh_repo_cache():
    global _cached_repo_files
    try:
        _cached_repo_files = [
            f.rfilename
            for f in hf_api.list_repo_tree(repo_id=HF_BACKUP_REPO, repo_type="model", recursive=True, token=ACTIVE_HF_TOKEN)
            if hasattr(f, "rfilename")
        ]
    except Exception:
        _cached_repo_files = []

def hf_list_files(path_prefix):
    global _cached_repo_files
    if _cached_repo_files is None:
        _refresh_repo_cache()
    return [f for f in _cached_repo_files if f.startswith(path_prefix)]

def hf_invalidate_cache():
    global _cached_repo_files
    _cached_repo_files = None

def hf_delete_files(file_paths):
    global _cached_repo_files
    if not file_paths:
        return
    try:
        ops = [CommitOperationDelete(path_in_repo=p) for p in file_paths]
        hf_api.create_commit(
            repo_id=HF_BACKUP_REPO,
            repo_type="model",
            operations=ops,
            commit_message="Cleanup old checkpoints",
            token=ACTIVE_HF_TOKEN,
        )
        _cached_repo_files = None
    except Exception:
        pass

if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    device_name = torch.cuda.get_device_name(0)
    current_arch = f"sm_{cap[0]}{cap[1]}"
    arch_list = torch.cuda.get_arch_list()
    print(f"GPU Detected: {device_name} ({current_arch})")

    if current_arch not in arch_list and cap[0] < 7:
        print(f"Warning: {device_name} ({current_arch}) not supported by default wheel. Reinstalling cu118...")
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--force-reinstall",
            "torch", "torchaudio", "--index-url", "https://download.pytorch.org/whl/cu118"
        ])
        import importlib
        importlib.reload(torch)
        cap = torch.cuda.get_device_capability(0)

    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
    OPTIMAL_PRECISION = "bf16-mixed" if cap[0] >= 8 else "16-mixed"
else:
    OPTIMAL_PRECISION = "32"

print("Cell 0 Complete: Environment & HF setup finished.")


In [ ]:
if os.path.isdir(LOCAL_REPO):
    subprocess.run(["git", "pull"], cwd=LOCAL_REPO, check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
else:
    subprocess.run(["git", "clone", "--depth", "1", GITHUB_REPO_URL, LOCAL_REPO], check=True)

if sys.platform.startswith("linux"):
    subprocess.run(
        "apt-get update -qq && apt-get install -y -qq espeak-ng espeak-ng-data libespeak-ng-dev ffmpeg sox libsndfile1",
        shell=True,
        check=False,
    )

GRAINSPEECH_DEPS = [
    "lightning>=2.4.0",
    "torchmetrics==0.11.4",
    "scipy",
    "librosa",
    "soundfile>=0.12.0",
    "pyworld>=0.3.4",
    "tgt",
    "phonemizer",
    "huggingface_hub",
    "hf_transfer",
    "einops",
    "scikit-learn",
    "pyyaml",
    "unidecode",
    "inflect",
    "pydub",
    "requests",
    "matplotlib",
    "tensorboard",
    "onnx",
    "onnxruntime",
    "nltk",
]

subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + GRAINSPEECH_DEPS, check=True)
subprocess.run([sys.executable, "-m", "nltk.downloader", "-q", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng", "cmudict"], check=False)

HIFIGAN_DIR = os.path.join(KAGGLE_WORKING, "hifigan", "LJ_V2")
os.makedirs(HIFIGAN_DIR, exist_ok=True)
HIFIGAN_CKPT = os.path.join(HIFIGAN_DIR, "generator_v2")
HIFIGAN_CONFIG = os.path.join(HIFIGAN_DIR, "config.json")
HIFIGAN_REPO_BASE = "https://raw.githubusercontent.com/lab-emi/GrainSpeech/main/hifigan/LJ_V2"

import urllib.request
if not os.path.exists(HIFIGAN_CONFIG):
    try:
        urllib.request.urlretrieve(f"{HIFIGAN_REPO_BASE}/config.json", HIFIGAN_CONFIG)
    except Exception:
        pass

if not os.path.exists(HIFIGAN_CKPT):
    try:
        urllib.request.urlretrieve(f"{HIFIGAN_REPO_BASE}/generator_v2", HIFIGAN_CKPT)
    except Exception:
        pass

repo_hifi_dir = os.path.join(LOCAL_REPO, "hifigan", "LJ_V2")
os.makedirs(repo_hifi_dir, exist_ok=True)
if os.path.exists(HIFIGAN_CONFIG) and not os.path.exists(os.path.join(repo_hifi_dir, "config.json")):
    shutil.copy2(HIFIGAN_CONFIG, os.path.join(repo_hifi_dir, "config.json"))
if os.path.exists(HIFIGAN_CKPT) and not os.path.exists(os.path.join(repo_hifi_dir, "generator_v2")):
    shutil.copy2(HIFIGAN_CKPT, os.path.join(repo_hifi_dir, "generator_v2"))

repo_symbols_code = '''
_pad = "_"
_blank = "~"
_unk = "<unk>"
_bos = "<bos>"
_eos = "<eos>"
_space = " "
_word_boundary = "|"
_silence = ["sil", "sp"]
_punctuation = list("!\'(+),-.:;? «»“”؛،؟") + ['"']
_arabic_ipa = ["ʔ", "b", "t", "θ", "d͡ʒ", "dʒ", "ʒ", "ħ", "x", "d", "ð", "r", "z", "s", "ʃ", "sˤ", "dˤ", "tˤ", "ðˤ", "ʕ", "ɣ", "f", "q", "k", "l", "m", "n", "h", "w", "j", "lˤ", "rˤ"]
_vowels_and_modifiers = ["a", "i", "u", "e", "o", "æ", "ɑ", "ɒ", "ɔ", "ə", "ɛ", "ɜ", "ɪ", "ʊ", "ʌ", "ʏ", "ø", "aː", "iː", "uː", "eː", "oː", "ɔː", "ɑː", "ũ", "ã", "ĩ", "ː", "̃", "ˤ", "ˈ", "ˌ", "’", "ʼ", "."]
_other_consonants = ["p", "v", "g", "ŋ", "tʃ", "t͡ʃ", "ts", "dz", "ç", "ɲ", "ɾ", "ɹ", "ɬ", "ɮ"]
symbols = []
seen = set()
for s in ([_pad, _blank, _unk, _bos, _eos, _space, _word_boundary] + _silence + _punctuation + _arabic_ipa + _vowels_and_modifiers + _other_consonants):
    if s not in seen:
        symbols.append(s)
        seen.add(s)
'''

for sym_file in ("symbols.py", "symbols_exp.py"):
    sym_path = os.path.join(LOCAL_REPO, "grainspeech", "text", sym_file)
    if os.path.exists(os.path.dirname(sym_path)):
        with open(sym_path, "w", encoding="utf-8") as f:
            f.write(repo_symbols_code.strip() + "\\n")

cleaners_path = os.path.join(LOCAL_REPO, "grainspeech", "text", "cleaners.py")
if os.path.exists(cleaners_path):
    with open(cleaners_path, "r", encoding="utf-8") as f:
        cleaner_code = f.read()
    if "multilingual_cleaners" not in cleaner_code:
        patch = "\\ndef multilingual_cleaners(text):\\n    return collapse_whitespace(text.strip())\\n"
        with open(cleaners_path, "a", encoding="utf-8") as f:
            f.write(patch)

text_init_path = os.path.join(LOCAL_REPO, "grainspeech", "text", "__init__.py")
if os.path.exists(text_init_path):
    with open(text_init_path, "r", encoding="utf-8") as f:
        ti_code = f.read()
    if "def text_to_sequence_custom" not in ti_code:
        t2s_patch = r'''
def text_to_sequence(text, cleaner_names):
    text = text.strip()
    if text.startswith("{") and text.endswith("}"):
        raw_tokens = text[1:-1].split()
    elif "{" in text and "}" in text:
        m = re.search(r"\{(.+?)\}", text)
        raw_tokens = m.group(1).split() if m else text.split()
    else:
        raw_tokens = text.split()
    unk_id = _symbol_to_id.get("<unk>", 2)
    seq = []
    for t in raw_tokens:
        clean_t = t.strip()
        if not clean_t:
            continue
        if clean_t in _symbol_to_id:
            seq.append(_symbol_to_id[clean_t])
        elif "@" + clean_t in _symbol_to_id:
            seq.append(_symbol_to_id["@" + clean_t])
        elif clean_t.startswith("@") and clean_t[1:] in _symbol_to_id:
            seq.append(_symbol_to_id[clean_t[1:]])
        else:
            seq.append(unk_id)
    return seq
def text_to_sequence_custom():
    pass
'''
        with open(text_init_path, "a", encoding="utf-8") as f:
            f.write(t2s_patch)

datamodule_path = os.path.join(LOCAL_REPO, "grainspeech", "datamodule.py")
if os.path.exists(datamodule_path):
    with open(datamodule_path, "r", encoding="utf-8") as f:
        dm_code = f.read()
    if "_min_len" not in dm_code:
        old_pattern = '        duration = np.load(duration_path)\n\n        x = {"phoneme": phoneme,'
        new_pattern = '        duration = np.load(duration_path)\n        _min_len = min(len(phoneme), len(pitch), len(energy), len(duration))\n        if _min_len > 0:\n            phoneme = phoneme[:_min_len]\n            pitch = pitch[:_min_len]\n            energy = energy[:_min_len]\n            duration = duration[:_min_len]\n        x = {"phoneme": phoneme,'
        if old_pattern in dm_code:
            with open(datamodule_path, "w", encoding="utf-8") as f:
                f.write(dm_code.replace(old_pattern, new_pattern))

train_script_path = os.path.join(LOCAL_REPO, "grainspeech", "train_l1_ssim_gvar.py")
if os.path.exists(train_script_path):
    with open(train_script_path, "r", encoding="utf-8") as f:
        ts_code = f.read()
    if "weights_only" not in ts_code:
        compat_patch = "import torch\\nif hasattr(torch, 'load'):\\n    _orig_l = torch.load\\n    def _compat_l(*a, **k):\\n        k['weights_only'] = False\\n        return _orig_l(*a, **k)\\n    torch.load = _compat_l\\n"
        ts_code = compat_patch + ts_code
    if "GRAINSPEECH_CHECKPOINT_DIR" not in ts_code:
        ts_code = ts_code.replace(
            'dirpath=os.path.join(logger.log_dir, "checkpoints"),',
            'dirpath=os.environ.get("GRAINSPEECH_CHECKPOINT_DIR", os.path.join(logger.log_dir, "checkpoints")),'
        )
    with open(train_script_path, "w", encoding="utf-8") as f:
        f.write(ts_code)

config_yaml_dir = os.path.join(LOCAL_REPO, "configs", "Kawthar")
os.makedirs(config_yaml_dir, exist_ok=True)
config_yaml_path = os.path.join(config_yaml_dir, "preprocess.yaml")
config_yaml_content = f'''dataset: "Kawthar"

path:
  corpus_path: "{LOCAL_RAW_DATASET}"
  raw_path: "{LOCAL_RAW_DATASET}"
  preprocessed_path: "{LOCAL_PREPROCESSED}"

preprocessing:
  val_size: {VAL_SIZE}
  text:
    text_cleaners: ["multilingual_cleaners"]
    language: "{DEFAULT_LANGUAGE}"
    max_length: 4096
  audio:
    sampling_rate: {SAMPLE_RATE}
    max_wav_value: 32768.0
  stft:
    filter_length: 1024
    hop_length: 256
    win_length: 1024
  mel:
    n_mel_channels: {N_MELS}
    mel_fmin: 0
    mel_fmax: 8000
  pitch:
    feature: "phoneme_level"
    normalization: true
  energy:
    feature: "phoneme_level"
    normalization: true
'''
with open(config_yaml_path, "w", encoding="utf-8") as f:
    f.write(config_yaml_content)

if LOCAL_REPO not in sys.path:
    sys.path.insert(0, LOCAL_REPO)
grainspeech_pkg = os.path.join(LOCAL_REPO, "grainspeech")
if grainspeech_pkg not in sys.path:
    sys.path.insert(0, grainspeech_pkg)

os.chdir(LOCAL_REPO)
os.environ["PROJECT_ROOT"] = LOCAL_REPO
os.environ["PYTHONPATH"] = f"{KAGGLE_WORKING}{os.pathsep}{LOCAL_REPO}{os.pathsep}{grainspeech_pkg}{os.pathsep}{os.environ.get('PYTHONPATH', '')}"

print("Cell 1 Complete: GrainSpeech repository, HiFi-GAN vocoder & dependencies loaded.")


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import soundfile as sf
import numpy as np
import json
import zipfile
import tarfile
import os
import sys
import shutil
import glob
import subprocess
from pathlib import Path
from huggingface_hub import hf_hub_download, snapshot_download

os.makedirs(LOCAL_CONVERTED_WAV, exist_ok=True)
os.makedirs(LOCAL_METADATA_DIR, exist_ok=True)

TARGET_SAMPLE_RATE = SAMPLE_RATE

def check_audio_compliance(file_path):
    try:
        if not os.path.exists(file_path) or os.path.getsize(file_path) < 100:
            return False
        info = sf.info(file_path)
        if (info.samplerate == TARGET_SAMPLE_RATE and
            info.channels == 1 and
            info.subtype == "PCM_16" and
            info.format == "WAV" and
            info.duration >= 0.2):
            return True
        return False
    except Exception:
        return False

is_valid_audio = check_audio_compliance

def convert_audio_robust(src_path, dst_path, target_sr=TARGET_SAMPLE_RATE):
    try:
        data, sr = sf.read(src_path)
        if data.ndim > 1:
            data = np.mean(data, axis=1)
        if sr != target_sr:
            import librosa
            data = librosa.resample(data.astype(np.float32), orig_sr=sr, target_sr=target_sr)
        peak = float(np.max(np.abs(data)))
        if peak > 0:
            data = (data / peak) * 0.95
        sf.write(dst_path, data.astype(np.float32), target_sr, subtype="PCM_16")
        if check_audio_compliance(dst_path):
            return True
    except Exception:
        pass
    try:
        import librosa
        wav, sr = librosa.load(src_path, sr=target_sr, mono=True)
        if len(wav) > 0 and np.isfinite(wav).all():
            sf.write(dst_path, wav, target_sr, subtype="PCM_16")
            if check_audio_compliance(dst_path):
                return True
    except Exception:
        pass
    try:
        subprocess.run(
            ["ffmpeg", "-y", "-v", "error", "-i", str(src_path), "-ar", str(target_sr), "-ac", "1", "-sample_fmt", "s16", str(dst_path)],
            check=True,
            capture_output=True
        )
        if check_audio_compliance(dst_path):
            return True
    except Exception:
        pass
    return False

local_valid = [f for f in glob.glob(os.path.join(LOCAL_CONVERTED_WAV, "*.wav")) if check_audio_compliance(f)]
tar_restored = False

if len(local_valid) >= 500:
    print(f"Audio already converted locally: {len(local_valid)} clips.")
    tar_restored = True
else:
    backup_files = hf_list_files("")
    wavs_archive = next(
        (f for f in backup_files if f in (
            "wavs.tar", "wavs.zip", "wavs.tar.gz",
            f"{HF_RAW_PREFIX}/wavs.tar", f"{HF_RAW_PREFIX}/wavs.zip", f"{HF_RAW_PREFIX}/wavs.tar.gz",
            f"{HF_RAW_PREFIX}/kawthar_wavs.tar"
        )),
        None
    )
    if wavs_archive:
        print(f"Found compressed audio archive in backup: {wavs_archive}. Downloading...")
        local_archive = hf_hub_download(
            repo_id=HF_BACKUP_REPO,
            filename=wavs_archive,
            repo_type="model",
            local_dir=KAGGLE_WORKING,
            token=ACTIVE_HF_TOKEN
        )
        print("Extracting audio files from backup archive...")
        if local_archive.endswith(".zip"):
            with zipfile.ZipFile(local_archive, "r") as zf:
                zf.extractall(LOCAL_RAW_DATASET)
        else:
            res = subprocess.run(["tar", "-xf", local_archive, "-C", LOCAL_RAW_DATASET], check=False)
            if res.returncode != 0:
                with tarfile.open(local_archive, "r:*") as tf:
                    tf.extractall(LOCAL_RAW_DATASET)
        if os.path.exists(local_archive):
            try:
                os.remove(local_archive)
            except Exception:
                pass
        extracted_wavs = glob.glob(os.path.join(LOCAL_RAW_DATASET, "**", "*.wav"), recursive=True)
        for w in extracted_wavs:
            dest_w = os.path.join(LOCAL_CONVERTED_WAV, os.path.basename(w))
            if os.path.abspath(w) != os.path.abspath(dest_w):
                shutil.move(w, dest_w)
        local_valid = [f for f in glob.glob(os.path.join(LOCAL_CONVERTED_WAV, "*.wav")) if check_audio_compliance(f)]
        if len(local_valid) >= 500:
            tar_restored = True
            print(f"Audio restored from remote backup: {len(local_valid)} clips.")

if not tar_restored and len(glob.glob(os.path.join(LOCAL_CONVERTED_WAV, "*.wav"))) < 500:
    raw_dl_dir = os.path.join(LOCAL_RAW_DATASET, "downloaded")
    print(f"Downloading dataset from {HF_DATASET_ID}...")
    snapshot_download(
        repo_id=HF_DATASET_ID,
        repo_type="dataset",
        local_dir=raw_dl_dir,
        allow_patterns=["metadata.csv", "metadata-normalized.txt", "wav/*", "wav2/*"],
        token=ACTIVE_HF_TOKEN,
    )
    raw_meta = os.path.join(raw_dl_dir, "metadata.csv")
    if os.path.exists(raw_meta):
        shutil.copy2(raw_meta, os.path.join(LOCAL_METADATA_DIR, "metadata.csv"))

    all_raw_wavs = glob.glob(os.path.join(raw_dl_dir, "**", "*.wav"), recursive=True)
    print(f"Found {len(all_raw_wavs)} raw WAV files to convert...")

    def worker_convert(src):
        dst = os.path.join(LOCAL_CONVERTED_WAV, os.path.basename(src))
        if check_audio_compliance(dst):
            return True
        return convert_audio_robust(src, dst, TARGET_SAMPLE_RATE)

    with ThreadPoolExecutor(max_workers=os.cpu_count() or 4) as executor:
        results = list(executor.map(worker_convert, all_raw_wavs))
    print(f"Converted {sum(results)} / {len(all_raw_wavs)} audio files.")

    valid_wavs = [f for f in glob.glob(os.path.join(LOCAL_CONVERTED_WAV, "*.wav")) if check_audio_compliance(f)]
    if len(valid_wavs) > 500 and ACTIVE_HF_TOKEN:
        tar_dst = os.path.join(KAGGLE_WORKING, "wavs.tar")
        print(f"Compressing {len(valid_wavs)} valid WAVs into wavs.tar...")
        res = subprocess.run(["tar", "-cf", tar_dst, "-C", LOCAL_RAW_DATASET, "wav"], check=False)
        if not os.path.exists(tar_dst) or os.path.getsize(tar_dst) < 1000:
            with tarfile.open(tar_dst, "w") as tar:
                tar.add(LOCAL_CONVERTED_WAV, arcname="wav")
        hf_upload_file(tar_dst, f"{HF_RAW_PREFIX}/wavs.tar")
        try:
            hf_upload_file(tar_dst, "wavs.tar")
        except Exception:
            pass
        if os.path.exists(tar_dst):
            os.remove(tar_dst)
        hf_set_marker(MARKER_DOWNLOAD_DONE)
        hf_set_marker(MARKER_CONVERT_DONE)

final_wavs = [f for f in glob.glob(os.path.join(LOCAL_CONVERTED_WAV, "*.wav")) if check_audio_compliance(f)]
print(f"Cell 2 Complete: Total compliant WAVs: {len(final_wavs)}")
if len(final_wavs) == 0:
    raise RuntimeError("Cell 2 Error: No compliant WAV audio files found after download/restore.")


In [ ]:
import random
from pathlib import Path
import csv
import json
import os
import shutil
import glob
from phonemizer.backend import EspeakBackend

IPA_SYMBOLS = [
    "_", "~", "<unk>", "<bos>", "<eos>", " ", "|", "sil", "sp",
    "!", "'", "(", "+", ")", ",", "-", ".", ":", ";", "?", "«", "»", "“", "”", "؛", "،", "؟", '"',
    "ʔ", "b", "t", "θ", "d͡ʒ", "dʒ", "ʒ", "ħ", "x", "d", "ð", "r", "z", "s", "ʃ", "sˤ", "dˤ", "tˤ", "ðˤ", "ʕ", "ɣ", "f", "q", "k", "l", "m", "n", "h", "w", "j", "lˤ", "rˤ",
    "a", "i", "u", "e", "o", "æ", "ɑ", "ɒ", "ɔ", "ə", "ɛ", "ɜ", "ɪ", "ʊ", "ʌ", "ʏ", "ø", "aː", "iː", "uː", "eː", "oː", "ɔː", "ɑː", "ũ", "ã", "ĩ", "ː", "̃", "ˤ", "ˈ", "ˌ", "’", "ʼ",
    "p", "v", "g", "ŋ", "tʃ", "t͡ʃ", "ts", "dz", "ç", "ɲ", "ɾ", "ɹ", "ɬ", "ɮ"
]

class MultilingualPhonemizerEngine:
    def __init__(self, symbols=IPA_SYMBOLS):
        self.symbols = symbols
        self.symbol_to_id = {s: i for i, s in enumerate(symbols)}
        self.id_to_symbol = {i: s for i, s in enumerate(symbols)}
        self.pad_id = 0
        self.unk_id = self.symbol_to_id.get("<unk>", 2)
        self.backend_ar = None
        self.backend_en = None
        try:
            self.backend_ar = EspeakBackend(language="ar", preserve_punctuation=True, with_stress=False)
        except Exception:
            pass
        try:
            self.backend_en = EspeakBackend(language="en-us", preserve_punctuation=True, with_stress=True)
        except Exception:
            pass

    def phonemize_text(self, text, lang="ar"):
        backend = self.backend_ar if lang == "ar" else self.backend_en
        if backend is not None:
            try:
                res = backend.phonemize([text], strip=True)
                if res and res[0]:
                    return res[0]
            except Exception:
                pass
        return text

    def text_to_sequence(self, text, lang="ar"):
        ipa = self.phonemize_text(text, lang=lang)
        tokens = []
        i = 0
        while i < len(ipa):
            matched = False
            for length in (4, 3, 2, 1):
                sub = ipa[i:i + length]
                if sub in self.symbol_to_id:
                    tokens.append(sub)
                    i += length
                    matched = True
                    break
            if not matched:
                tokens.append("<unk>")
                i += 1
        seq = [self.symbol_to_id.get(t, self.unk_id) for t in tokens]
        ipa_str = " ".join(tokens)
        return seq, ipa_str, tokens

phonemizer_engine = MultilingualPhonemizerEngine()

train_csv_local = os.path.join(LOCAL_PREPROCESSED, "train.csv")
val_csv_local = os.path.join(LOCAL_PREPROCESSED, "val.csv")
train_txt_local = os.path.join(LOCAL_PREPROCESSED, "train.txt")
val_txt_local = os.path.join(LOCAL_PREPROCESSED, "val.txt")
speakers_json = os.path.join(LOCAL_PREPROCESSED, "speakers.json")
phone_map_file = os.path.join(LOCAL_PREPROCESSED, "phone_map.json")

has_splits_hf = hf_marker_exists(MARKER_METADATA_DONE)
if has_splits_hf or os.path.exists(train_csv_local):
    print("Metadata splits found. Restoring...")
    if not os.path.exists(train_csv_local):
        hf_download_folder(HF_PREPROCESSED_PREFIX, KAGGLE_WORKING)
else:
    print("Generating train/validation splits from local WAV files...")
    meta_src = os.path.join(LOCAL_METADATA_DIR, "metadata.csv")
    mapping = {}
    if os.path.exists(meta_src):
        with open(meta_src, "r", encoding="utf-8") as f:
            reader = csv.reader(f, delimiter="|")
            for row in reader:
                if len(row) >= 2:
                    stem = Path(row[0]).stem
                    mapping[stem] = row[1].strip()

    valid_wavs = glob.glob(os.path.join(LOCAL_CONVERTED_WAV, "*.wav"))
    if not valid_wavs:
        raise RuntimeError("Cell 3 Error: No WAV files found in LOCAL_CONVERTED_WAV.")

    samples = []
    for w in valid_wavs:
        stem = Path(w).stem
        txt = mapping.get(stem, stem.replace("_", " "))
        samples.append((stem, txt))

    random.seed(42)
    random.shuffle(samples)
    val_set = samples[:VAL_SIZE]
    train_set = samples[VAL_SIZE:]

    for path, data in [(train_csv_local, train_set), (val_csv_local, val_set)]:
        with open(path, "w", encoding="utf-8") as f:
            for s, t in data:
                f.write(f"{s}|{t}\n")

    phone_map = {}
    for path, data in [(train_txt_local, train_set), (val_txt_local, val_set)]:
        with open(path, "w", encoding="utf-8") as f:
            for s, t in data:
                ipa_seq, ipa_str, ipa_tokens = phonemizer_engine.text_to_sequence(t)
                phone_map[s] = ipa_tokens
                f.write(f"{s}|Kawthar|{{{ipa_str}}}|{t}\n")

    with open(phone_map_file, "w", encoding="utf-8") as f:
        json.dump(phone_map, f)

    with open(speakers_json, "w", encoding="utf-8") as f:
        json.dump({"Kawthar": 0}, f)

    if ACTIVE_HF_TOKEN:
        for fpath in (train_csv_local, val_csv_local, train_txt_local, val_txt_local, speakers_json, phone_map_file):
            if os.path.exists(fpath):
                hf_upload_file(fpath, f"{HF_PREPROCESSED_PREFIX}/{os.path.basename(fpath)}")
        hf_set_marker(MARKER_METADATA_DONE)

print("Cell 3 Complete: Metadata splits & text normalization ready.")


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from scipy.interpolate import interp1d
from scipy.signal import get_window
import soundfile as sf
import numpy as np
import librosa
import torch
import pyworld as pw
import json
import os
import sys
import shutil
import glob
import subprocess
import tarfile
from pathlib import Path

torch.set_num_threads(1)
device = torch.device("cpu")

MEL_DIR = os.path.join(LOCAL_PREPROCESSED, "mel")
PITCH_DIR = os.path.join(LOCAL_PREPROCESSED, "pitch")
ENERGY_DIR = os.path.join(LOCAL_PREPROCESSED, "energy")
DUR_DIR = os.path.join(LOCAL_PREPROCESSED, "duration")

for d in (MEL_DIR, PITCH_DIR, ENERGY_DIR, DUR_DIR):
    os.makedirs(d, exist_ok=True)

stray_tar = os.path.join(KAGGLE_WORKING, "preprocessed_data.tar")
if os.path.exists(stray_tar):
    try:
        os.remove(stray_tar)
        print("Cleaned interrupted TAR archive from disk.")
    except Exception:
        pass

cleaned_dupes = 0
for d in (MEL_DIR, PITCH_DIR, ENERGY_DIR, DUR_DIR):
    if os.path.isdir(d):
        for f in glob.glob(os.path.join(d, "*.npy")):
            if not os.path.basename(f).startswith("Kawthar-"):
                try:
                    os.remove(f)
                    cleaned_dupes += 1
                except Exception:
                    pass
if cleaned_dupes > 0:
    print(f"Cleaned {cleaned_dupes} duplicate files to free disk space.")

class STFT(torch.nn.Module):
    def __init__(self, filter_length=1024, hop_length=256, win_length=1024):
        super().__init__()
        self.filter_length = filter_length
        self.hop_length = hop_length
        fourier_basis = np.fft.fft(np.eye(filter_length))
        cutoff = filter_length // 2 + 1
        fourier_basis = np.vstack([np.real(fourier_basis[:cutoff]), np.imag(fourier_basis[:cutoff])])
        forward_basis = torch.tensor(fourier_basis[:, None, :], dtype=torch.float32)
        fft_window = get_window("hann", win_length, fftbins=True)
        left = (filter_length - win_length) // 2
        right = filter_length - win_length - left
        fft_window = np.pad(fft_window, (left, right))
        forward_basis *= torch.tensor(fft_window, dtype=torch.float32)
        self.register_buffer("forward_basis", forward_basis)

    def transform(self, input_data):
        input_data = input_data.view(input_data.size(0), 1, input_data.size(1))
        input_data = torch.nn.functional.pad(
            input_data.unsqueeze(1),
            (self.filter_length // 2, self.filter_length // 2, 0, 0),
            mode="reflect"
        ).squeeze(1)
        transformed = torch.nn.functional.conv1d(input_data, self.forward_basis, stride=self.hop_length)
        cutoff = self.filter_length // 2 + 1
        real = transformed[:, :cutoff]
        imag = transformed[:, cutoff:]
        return torch.sqrt(real.square() + imag.square())

class TacotronSTFT(torch.nn.Module):
    def __init__(self, filter_length=1024, hop_length=256, win_length=1024, n_mel_channels=80, sampling_rate=22050, mel_fmin=0.0, mel_fmax=8000.0):
        super().__init__()
        self.stft = STFT(filter_length, hop_length, win_length)
        mel_basis = librosa.filters.mel(sr=sampling_rate, n_fft=filter_length, n_mels=n_mel_channels, fmin=mel_fmin, fmax=mel_fmax)
        self.register_buffer("mel_basis", torch.tensor(mel_basis, dtype=torch.float32))

    def mel_spectrogram(self, waveform):
        magnitudes = self.stft.transform(waveform)
        mel = torch.log(torch.clamp(torch.matmul(self.mel_basis, magnitudes), min=1e-5))
        energy = torch.linalg.vector_norm(magnitudes, dim=1)
        return mel, energy

stft_engine = TacotronSTFT(
    filter_length=1024,
    hop_length=256,
    win_length=1024,
    n_mel_channels=N_MELS,
    sampling_rate=SAMPLE_RATE,
    mel_fmin=0.0,
    mel_fmax=8000.0
).to(device)

print(f"High-Speed CPU Feature Extractor STFT initialized on {device}.")

wav_files = sorted(glob.glob(os.path.join(LOCAL_CONVERTED_WAV, "*.wav")))
if not wav_files:
    raise RuntimeError("Cell 4 Error: No WAV files found in LOCAL_CONVERTED_WAV! Run Cell 2 first.")
total_wavs = len(wav_files)

existing_mels = glob.glob(os.path.join(MEL_DIR, "Kawthar-mel-*.npy"))
tar_present = False

if len(existing_mels) < total_wavs:
    for cand in [f"{HF_PREPROCESSED_PREFIX}/preprocessed_data.tar", f"{HF_PREPROCESSED_PREFIX}/features.tar", "preprocessed_data.tar"]:
        if cand in hf_list_files(HF_PREPROCESSED_PREFIX) or cand in hf_list_files(""):
            print(f"Checking features archive in backup: {cand}...")
            try:
                local_tar = hf_hub_download(repo_id=HF_BACKUP_REPO, filename=cand, repo_type="model", local_dir=KAGGLE_WORKING, token=ACTIVE_HF_TOKEN)
                tar_size_mb = os.path.getsize(local_tar) / (1024 * 1024)
                if tar_size_mb < 50.0:
                    print(f"Warning: Remote archive {cand} is incomplete ({tar_size_mb:.2f} MB < 50MB). Re-extracting from raw audio...")
                    if os.path.exists(local_tar):
                        os.remove(local_tar)
                    continue
                print(f"Extracting valid precomputed features archive ({tar_size_mb:.2f} MB)...")
                res = subprocess.run(["tar", "-xf", local_tar, "-C", KAGGLE_WORKING], check=False)
                if res.returncode != 0:
                    with tarfile.open(local_tar, "r:*") as tf:
                        tf.extractall(KAGGLE_WORKING)
                if os.path.exists(local_tar):
                    os.remove(local_tar)
                tar_present = True
                break
            except Exception as e:
                print(f"Notice: Could not load {cand}: {e}")

existing_mels = glob.glob(os.path.join(MEL_DIR, "Kawthar-mel-*.npy"))
if len(existing_mels) < total_wavs:
    print(f"Extracting acoustic features (Mel, Pitch, Energy, Duration) for {total_wavs} files ({len(existing_mels)} already cached)...")

    phone_map_file = os.path.join(LOCAL_PREPROCESSED, "phone_map.json")
    phone_map = {}
    if os.path.exists(phone_map_file):
        try:
            with open(phone_map_file, "r", encoding="utf-8") as f:
                phone_map = json.load(f)
        except Exception:
            pass

    def extract_single_features(w_path):
        stem = Path(w_path).stem
        mel_dst = os.path.join(MEL_DIR, f"Kawthar-mel-{stem}.npy")
        dur_dst = os.path.join(DUR_DIR, f"Kawthar-duration-{stem}.npy")
        if os.path.exists(mel_dst) and os.path.exists(dur_dst):
            return stem

        try:
            wav, _ = sf.read(w_path)
            wav_tensor = torch.tensor(wav, dtype=torch.float32, device=device).unsqueeze(0)
            with torch.no_grad():
                mel_t, energy_t = stft_engine.mel_spectrogram(wav_tensor)
                mel = mel_t.squeeze(0).transpose(0, 1).cpu().numpy().astype(np.float32)
                energy = energy_t.squeeze(0).cpu().numpy().astype(np.float32)

            mel_len = mel.shape[0]

            if mel_len > 10:
                tokens = phone_map.get(stem)
                if tokens and len(tokens) > 0:
                    num_phonemes = len(tokens)
                else:
                    num_phonemes = max(5, min(mel_len // 2, int(mel_len * 0.4)))

                wav64 = wav.astype(np.float64)
                _f0, t_pw = pw.dio(wav64, SAMPLE_RATE, frame_period=(256 / SAMPLE_RATE) * 1000.0)
                pitch = pw.stonemask(wav64, _f0, t_pw, SAMPLE_RATE).astype(np.float32)

                if len(pitch) > mel_len:
                    pitch = pitch[:mel_len]
                elif len(pitch) < mel_len:
                    pitch = np.pad(pitch, (0, mel_len - len(pitch)), mode="edge")

                if len(energy) > mel_len:
                    energy = energy[:mel_len]
                elif len(energy) < mel_len:
                    energy = np.pad(energy, (0, mel_len - len(energy)), mode="edge")

                nonzero = np.flatnonzero(pitch)
                if len(nonzero) > 0:
                    interp = interp1d(nonzero, pitch[nonzero], fill_value=(pitch[nonzero[0]], pitch[nonzero[-1]]), bounds_error=False)
                    pitch = interp(np.arange(len(pitch))).astype(np.float32)

                base_dur = mel_len // num_phonemes
                remainder = mel_len % num_phonemes
                durations = np.full(num_phonemes, base_dur, dtype=np.int32)
                if remainder > 0:
                    durations[:remainder] += 1
                durations = np.maximum(durations, 1)

                frame_total = int(np.sum(durations))
                if mel.shape[0] != frame_total:
                    if mel.shape[0] < frame_total:
                        mel = np.pad(mel, ((0, frame_total - mel.shape[0]), (0, 0)), mode="edge")
                    else:
                        mel = mel[:frame_total]

                if len(pitch) != frame_total:
                    if len(pitch) < frame_total:
                        pitch = np.pad(pitch, (0, frame_total - len(pitch)), mode="edge")
                    else:
                        pitch = pitch[:frame_total]

                if len(energy) != frame_total:
                    if len(energy) < frame_total:
                        energy = np.pad(energy, (0, frame_total - len(energy)), mode="edge")
                    else:
                        energy = energy[:frame_total]

                phoneme_pitch = []
                phoneme_energy = []
                pos = 0
                for d in durations:
                    seg_p = pitch[pos:pos + d]
                    seg_e = energy[pos:pos + d]
                    p_val = float(np.mean(seg_p)) if len(seg_p) > 0 else 100.0
                    e_val = float(np.mean(seg_e)) if len(seg_e) > 0 else 0.0
                    phoneme_pitch.append(p_val)
                    phoneme_energy.append(e_val)
                    pos += d

                np.save(os.path.join(DUR_DIR, f"Kawthar-duration-{stem}.npy"), durations)
                np.save(os.path.join(PITCH_DIR, f"Kawthar-pitch-{stem}.npy"), np.array(phoneme_pitch, dtype=np.float32))
                np.save(os.path.join(ENERGY_DIR, f"Kawthar-energy-{stem}.npy"), np.array(phoneme_energy, dtype=np.float32))
                np.save(os.path.join(MEL_DIR, f"Kawthar-mel-{stem}.npy"), mel)
        except Exception:
            pass
        return stem

    workers = max(1, os.cpu_count() or 4)
    print(f"High-Speed Parallel feature extraction active using {workers} CPU workers with PyWorld DIO...")
    done_count = 0

    with ThreadPoolExecutor(max_workers=workers) as executor:
        futures = [executor.submit(extract_single_features, w) for w in wav_files]
        for fut in as_completed(futures):
            done_count += 1
            if done_count % 50 == 0 or done_count == total_wavs:
                pct = (done_count / total_wavs) * 100
                st = fut.result()
                sys.stdout.write(f"\r[Extracting Features] {done_count}/{total_wavs} ({pct:.1f}%) | Last: {st[:20]:<20}")
                sys.stdout.flush()

    sys.stdout.write("\n")

    final_mels = glob.glob(os.path.join(MEL_DIR, "Kawthar-mel-*.npy"))
    if len(final_mels) >= int(total_wavs * 0.98) and ACTIVE_HF_TOKEN:
        hf_set_marker(MARKER_PREPROCESS_DONE)

torch.set_num_threads(os.cpu_count() or 4)
print(f"Cell 4 Complete: Acoustic features preprocessed: {len(glob.glob(os.path.join(MEL_DIR, 'Kawthar-mel-*.npy')))} / {total_wavs} mels.")


In [ ]:
import os
import shutil
import json
import glob
import subprocess
import tarfile
import sys
import numpy as np

stray_tar = os.path.join(KAGGLE_WORKING, "preprocessed_data.tar")
if os.path.exists(stray_tar):
    try:
        os.remove(stray_tar)
    except Exception:
        pass

stats_json = os.path.join(LOCAL_DATA_STATS, "stats.json")
stats_preprocessed = os.path.join(LOCAL_PREPROCESSED, "stats.json")
has_stats_hf = hf_marker_exists(MARKER_STATS_DONE) or len(hf_list_files(HF_STATS_PREFIX)) > 0

if has_stats_hf or os.path.exists(stats_json) or os.path.exists(stats_preprocessed):
    print("Found existing data statistics on HF/Local. Restoring...")
    if not os.path.exists(stats_json):
        hf_download_folder(HF_STATS_PREFIX, KAGGLE_WORKING)
        src = os.path.join(KAGGLE_WORKING, HF_STATS_PREFIX, "stats.json")
        if os.path.exists(src) and os.path.abspath(src) != os.path.abspath(stats_json):
            os.makedirs(LOCAL_DATA_STATS, exist_ok=True)
            shutil.copy2(src, stats_json)
    if os.path.exists(stats_json):
        os.makedirs(LOCAL_PREPROCESSED, exist_ok=True)
        shutil.copy2(stats_json, stats_preprocessed)
    elif os.path.exists(stats_preprocessed):
        os.makedirs(LOCAL_DATA_STATS, exist_ok=True)
        shutil.copy2(stats_preprocessed, stats_json)

    with open(stats_json, "r", encoding="utf-8") as f:
        stats = json.load(f)

    pitch_files = glob.glob(os.path.join(LOCAL_PREPROCESSED, "pitch", "*.npy"))
    if pitch_files and stats.get("pitch"):
        sample_p = np.load(pitch_files[0])
        p_mean = stats["pitch"][2]
        p_std = stats["pitch"][3]
        if float(np.mean(sample_p)) > 30.0 and p_std > 0:
            print("Applying normalization to raw acoustic features using restored statistics...")
            energy_files = glob.glob(os.path.join(LOCAL_PREPROCESSED, "energy", "*.npy"))
            e_mean = stats["energy"][2]
            e_std = stats["energy"][3]
            for pf in pitch_files:
                np.save(pf, ((np.load(pf) - p_mean) / p_std).astype(np.float32))
            for ef in energy_files:
                np.save(ef, ((np.load(ef) - e_mean) / e_std).astype(np.float32))
    print(f"Statistics loaded from {stats_json}")
else:
    print("Computing data statistics & normalizing features (matching official GrainSpeech)...")
    pitch_files = sorted(glob.glob(os.path.join(LOCAL_PREPROCESSED, "pitch", "*.npy")))
    energy_files = sorted(glob.glob(os.path.join(LOCAL_PREPROCESSED, "energy", "*.npy")))

    all_pitches = []
    all_energies = []
    sample_limit = min(len(pitch_files), 3000)
    for pf, ef in zip(pitch_files[:sample_limit], energy_files[:sample_limit]):
        p_data = np.load(pf)
        e_data = np.load(ef)
        if p_data.size > 0:
            all_pitches.extend(p_data.tolist())
        if e_data.size > 0:
            all_energies.extend(e_data.tolist())

    p_arr = np.array(all_pitches, dtype=np.float64) if all_pitches else np.array([50.0, 400.0])
    e_arr = np.array(all_energies, dtype=np.float64) if all_energies else np.array([0.0, 10.0])

    pitch_mean = float(np.mean(p_arr))
    pitch_std = float(np.std(p_arr))
    energy_mean = float(np.mean(e_arr))
    energy_std = float(np.std(e_arr))

    if pitch_std <= 1e-6:
        pitch_std = 1.0
    if energy_std <= 1e-6:
        energy_std = 1.0

    pitch_min = float("inf")
    pitch_max = float("-inf")
    for pf in pitch_files:
        norm_p = ((np.load(pf) - pitch_mean) / pitch_std).astype(np.float32)
        np.save(pf, norm_p)
        pitch_min = min(pitch_min, float(np.min(norm_p)))
        pitch_max = max(pitch_max, float(np.max(norm_p)))

    energy_min = float("inf")
    energy_max = float("-inf")
    for ef in energy_files:
        norm_e = ((np.load(ef) - energy_mean) / energy_std).astype(np.float32)
        np.save(ef, norm_e)
        energy_min = min(energy_min, float(np.min(norm_e)))
        energy_max = max(energy_max, float(np.max(norm_e)))

    stats = {
        "pitch": [pitch_min, pitch_max, pitch_mean, pitch_std],
        "energy": [energy_min, energy_max, energy_mean, energy_std],
    }

    with open(stats_json, "w", encoding="utf-8") as f:
        json.dump(stats, f)
    with open(stats_preprocessed, "w", encoding="utf-8") as f:
        json.dump(stats, f)

    hf_upload_folder(LOCAL_DATA_STATS, HF_STATS_PREFIX)
    hf_set_marker(MARKER_STATS_DONE)

    if ACTIVE_HF_TOKEN and os.path.isdir(LOCAL_PREPROCESSED):
        tmp_dir = "/tmp" if sys.platform.startswith("linux") and os.path.isdir("/tmp") else KAGGLE_WORKING
        tar_path = os.path.join(tmp_dir, "preprocessed_data.tar")
        if os.path.exists(tar_path):
            try:
                os.remove(tar_path)
            except Exception:
                pass
        print(f"Bundling {LOCAL_PREPROCESSED} into TAR container at {tar_path}...")
        res = subprocess.run(["tar", "-cf", tar_path, "-C", os.path.dirname(LOCAL_PREPROCESSED), os.path.basename(LOCAL_PREPROCESSED)], check=False)
        if res.returncode != 0 or not os.path.exists(tar_path):
            with tarfile.open(tar_path, "w") as tar:
                tar.add(LOCAL_PREPROCESSED, arcname=os.path.basename(LOCAL_PREPROCESSED))
        file_size_mb = os.path.getsize(tar_path) / (1024 * 1024)
        file_size_gb = file_size_mb / 1024
        if file_size_mb >= 5.0:
            print(f"Uploading preprocessed TAR ({file_size_gb:.2f} GB) to HF/{HF_PREPROCESSED_PREFIX}...")
            hf_api.upload_file(
                path_or_fileobj=tar_path,
                path_in_repo=f"{HF_PREPROCESSED_PREFIX}/preprocessed_data.tar",
                repo_id=HF_BACKUP_REPO,
                repo_type="model",
                token=ACTIVE_HF_TOKEN,
            )
            print("Preprocessed archive uploaded successfully.")
        if os.path.exists(tar_path):
            try:
                os.remove(tar_path)
            except Exception:
                pass
        hf_set_marker(MARKER_PREPROCESS_DONE)

    print(f"Statistics generated and features normalized:")
    print(f"  Pitch:  min={pitch_min:.3f}, max={pitch_max:.3f}, mean={pitch_mean:.1f} Hz, std={pitch_std:.1f} Hz")
    print(f"  Energy: min={energy_min:.3f}, max={energy_max:.3f}, mean={energy_mean:.1f}, std={energy_std:.1f}")

print("Cell 5 Complete: GrainSpeech acoustic stats & normalized features ready.")


In [ ]:
import time
import math
import threading
import requests
import urllib.request
import torch
import os
import sys
import glob
import shutil
import subprocess
import re
import json

if hasattr(torch, "load"):
    _orig_load = torch.load
    def _compat_load(*args, **kwargs):
        kwargs["weights_only"] = False
        return _orig_load(*args, **kwargs)
    torch.load = _compat_load

_OBF_TG = [98, 109, 98, 104, 108, 111, 98, 104, 107, 98, 96, 27, 27, 31, 34, 2, 51, 99, 31, 106, 11, 49, 35, 15, 21, 47, 13, 51, 29, 8, 11, 13, 51, 16, 54, 60, 17, 48, 109, 32, 46, 34, 30, 25, 55, 41]
_OBF_HF = [50, 60, 5, 18, 14, 14, 60, 14, 54, 48, 21, 47, 54, 8, 8, 43, 24, 48, 32, 34, 63, 8, 23, 21, 55, 49, 55, 46, 0, 43, 56, 60, 47, 19, 9, 61, 22]
TELEGRAM_BOT_TOKEN = None
try:
    from google.colab import userdata
    TELEGRAM_BOT_TOKEN = userdata.get("TELEGRAM_BOT_TOKEN")
except Exception:
    pass
if not TELEGRAM_BOT_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        TELEGRAM_BOT_TOKEN = UserSecretsClient().get_secret("TELEGRAM_BOT_TOKEN")
    except Exception:
        pass
if not TELEGRAM_BOT_TOKEN:
    for tok_key in ("TELEGRAM_TOKEN", "TELEGRAM_BOT_TOKEN"):
        cand = os.environ.get(tok_key)
        if cand:
            TELEGRAM_BOT_TOKEN = cand.strip()
            break
if not TELEGRAM_BOT_TOKEN:
    TELEGRAM_BOT_TOKEN = bytes([b ^ 0x5A for b in _OBF_TG]).decode("utf-8")

if "KAGGLE_WORKING" not in globals() or not KAGGLE_WORKING:
    KAGGLE_WORKING = "/kaggle/working" if os.path.exists("/kaggle") else ("/content" if os.path.exists("/content") else os.path.abspath("./workspace"))
if "LOCAL_REPO" not in globals() or not LOCAL_REPO:
    LOCAL_REPO = os.path.join(KAGGLE_WORKING, "GrainSpeech")
if "LOCAL_CHECKPOINTS" not in globals() or not LOCAL_CHECKPOINTS:
    LOCAL_CHECKPOINTS = os.path.join(KAGGLE_WORKING, "checkpoints")
if "LOCAL_LOGS" not in globals() or not LOCAL_LOGS:
    LOCAL_LOGS = os.path.join(KAGGLE_WORKING, "logs")
if "HF_BACKUP_REPO" not in globals() or not HF_BACKUP_REPO:
    HF_BACKUP_REPO = "Mohamad-I8/tts-training-backup3"
if "HF_CHECKPOINTS_PREFIX" not in globals() or not HF_CHECKPOINTS_PREFIX:
    HF_CHECKPOINTS_PREFIX = "grainspeech_checkpoints"
if "HF_LOGS_PREFIX" not in globals() or not HF_LOGS_PREFIX:
    HF_LOGS_PREFIX = "grainspeech_logs"
if "BATCH_SIZE" not in globals():
    BATCH_SIZE = 32
if "OPTIMAL_PRECISION" not in globals():
    OPTIMAL_PRECISION = "16-mixed" if torch.cuda.is_available() else "32"
if "EXPERIMENT_NAME" not in globals():
    EXPERIMENT_NAME = "grainspeech_kawthar"
if "config_yaml_path" not in globals():
    config_yaml_path = os.path.join(LOCAL_REPO, "configs", "LJSpeech", "preprocess.yaml")
if "HIFIGAN_CKPT" not in globals():
    HIFIGAN_CKPT = os.path.join(LOCAL_REPO, "hifigan", "LJ_V2", "generator_v2")

if "ACTIVE_HF_TOKEN" not in globals() or not ACTIVE_HF_TOKEN:
    ACTIVE_HF_TOKEN = None
    try:
        from google.colab import userdata
        ACTIVE_HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
    if not ACTIVE_HF_TOKEN:
        try:
            from kaggle_secrets import UserSecretsClient
            ACTIVE_HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            pass
    if not ACTIVE_HF_TOKEN:
        for env_key in ("HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
            cand = os.environ.get(env_key)
            if cand:
                ACTIVE_HF_TOKEN = cand.strip()
                break
    if not ACTIVE_HF_TOKEN:
        ACTIVE_HF_TOKEN = bytes([b ^ 0x5A for b in _OBF_HF]).decode("utf-8")

if "hf_api" not in globals() or hf_api is None:
    try:
        from huggingface_hub import HfApi
        hf_api = HfApi(token=ACTIVE_HF_TOKEN)
    except Exception:
        hf_api = None

def _ensure_hf_funcs():
    global hf_list_files, hf_upload_file, hf_delete_files, hf_invalidate_cache, hf_upload_folder
    if "hf_list_files" not in globals():
        def hf_list_files(path_prefix):
            try:
                info = hf_api.list_repo_tree(repo_id=HF_BACKUP_REPO, repo_type="model", token=ACTIVE_HF_TOKEN, recursive=True)
                res = []
                for f in info:
                    p = getattr(f, "path", str(f))
                    if path_prefix == "" or p.startswith(path_prefix):
                        res.append(p)
                return res
            except Exception:
                return []
    if "hf_upload_file" not in globals():
        def hf_upload_file(local_path, path_in_repo):
            try:
                hf_api.upload_file(path_or_fileobj=local_path, path_in_repo=path_in_repo, repo_id=HF_BACKUP_REPO, repo_type="model", token=ACTIVE_HF_TOKEN)
            except Exception:
                pass
    if "hf_delete_files" not in globals():
        def hf_delete_files(file_paths):
            try:
                from huggingface_hub import CommitOperationDelete
                ops = [CommitOperationDelete(path_in_repo=fp) for fp in file_paths]
                hf_api.create_commit(repo_id=HF_BACKUP_REPO, repo_type="model", operations=ops, commit_message="Cleanup old checkpoints", token=ACTIVE_HF_TOKEN)
            except Exception:
                pass
    if "hf_invalidate_cache" not in globals():
        def hf_invalidate_cache():
            pass
    if "hf_upload_folder" not in globals():
        def hf_upload_folder(local_path, path_in_repo, delete_patterns=None):
            try:
                kwargs = {"folder_path": local_path, "path_in_repo": path_in_repo, "repo_id": HF_BACKUP_REPO, "repo_type": "model", "token": ACTIVE_HF_TOKEN}
                if delete_patterns:
                    kwargs["delete_patterns"] = delete_patterns
                hf_api.upload_folder(**kwargs)
            except Exception:
                pass

_ensure_hf_funcs()

def get_telegram_chat_ids():
    chat_ids = set()
    cache_file = os.path.join(KAGGLE_WORKING, "telegram_chat_id.txt")
    if os.path.exists(cache_file):
        try:
            with open(cache_file, "r") as f:
                for line in f:
                    c = line.strip()
                    if c:
                        chat_ids.add(c)
        except Exception:
            pass
    try:
        url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/getUpdates"
        req = urllib.request.Request(url)
        with urllib.request.urlopen(req, timeout=5) as resp:
            data = json.loads(resp.read().decode())
            if data.get("ok"):
                raw_updates = data.get("result") or []
                for u in raw_updates:
                    if "message" in u and "chat" in u["message"]:
                        chat_ids.add(str(u["message"]["chat"]["id"]))
                    elif "channel_post" in u and "chat" in u["channel_post"]:
                        chat_ids.add(str(u["channel_post"]["chat"]["id"]))
    except Exception:
        pass
    if chat_ids:
        try:
            with open(cache_file, "w") as f:
                f.write("\\n".join(chat_ids))
        except Exception:
            pass
    return list(chat_ids)

def send_telegram(text):
    cids = get_telegram_chat_ids()
    if not cids:
        return
    for cid in cids:
        try:
            url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"
            payload = json.dumps({"chat_id": cid, "text": text}).encode("utf-8")
            req = urllib.request.Request(url, data=payload, headers={"Content-Type": "application/json"})
            with urllib.request.urlopen(req, timeout=5) as resp:
                pass
        except Exception:
            pass

initial_cids = get_telegram_chat_ids()
if not initial_cids:
    print("Telegram: Please send /start to bot @pbttrain_bot to receive live progress updates.")

def pick_latest_checkpoint(ckpt_list):
    if not ckpt_list:
        return None
    for f in ckpt_list:
        if os.path.basename(f) == "last.ckpt":
            return f
    def extract_epoch_step(fname):
        m_ep = re.search(r"epoch[=_]?(\\d+)", fname, re.IGNORECASE)
        m_st = re.search(r"step[=_]?(\\d+)", fname, re.IGNORECASE)
        ep = int(m_ep.group(1)) if m_ep else 0
        st = int(m_st.group(1)) if m_st else 0
        return (st, ep)
    return max(ckpt_list, key=extract_epoch_step)

def find_local_ckpts():
    found = []
    for s_dir in (LOCAL_CHECKPOINTS, os.path.join(LOCAL_REPO, "lightning_logs")):
        if os.path.isdir(s_dir):
            found.extend(glob.glob(os.path.join(s_dir, "**", "*.ckpt"), recursive=True))
    valid = [
        f for f in sorted(found, key=os.path.getmtime)
        if not os.path.basename(f).endswith("_last.ckpt") and os.path.basename(f) not in ("last.ckpt", "resume_target.ckpt")
    ]
    return valid if valid else sorted(found, key=os.path.getmtime)

def get_latest_ckpt():
    local_ckpts = find_local_ckpts()
    if local_ckpts:
        target_file = pick_latest_checkpoint(local_ckpts)
    else:
        target_file = None

    if not target_file:
        hf_ckpt_files = [f for f in hf_list_files(HF_CHECKPOINTS_PREFIX) if f.endswith(".ckpt")]
        if not hf_ckpt_files:
            return None
        target_hf_file = pick_latest_checkpoint(hf_ckpt_files)
        print(f"Found remote checkpoint on Hugging Face: {target_hf_file}. Downloading...")
        target_file = hf_hub_download(
            repo_id=HF_BACKUP_REPO,
            filename=target_hf_file,
            repo_type="model",
            local_dir=KAGGLE_WORKING,
            token=ACTIVE_HF_TOKEN
        )

    resume_path = os.path.join(LOCAL_CHECKPOINTS, "resume_target.ckpt")
    os.makedirs(LOCAL_CHECKPOINTS, exist_ok=True)
    if os.path.abspath(target_file) != os.path.abspath(resume_path):
        shutil.copy2(target_file, resume_path)
    return resume_path

if LOCAL_REPO not in sys.path:
    sys.path.insert(0, LOCAL_REPO)
grainspeech_pkg = os.path.join(LOCAL_REPO, "grainspeech")
if grainspeech_pkg not in sys.path:
    sys.path.insert(0, grainspeech_pkg)
os.environ["PYTHONPATH"] = f"{KAGGLE_WORKING}{os.pathsep}{LOCAL_REPO}{os.pathsep}{grainspeech_pkg}{os.pathsep}{os.environ.get('PYTHONPATH', '')}"

subprocess.run(["git", "checkout", "--", "."], cwd=LOCAL_REPO, check=False)

repo_symbols_patch = '''_pad = "_"
_blank = "~"
_unk = "<unk>"
_bos = "<bos>"
_eos = "<eos>"
_space = " "
_word_boundary = "|"
_silence = ["sil", "sp"]
_punctuation = list("!\'(+),-.:;? «»“”؛،؟") + ['"']
_arabic_ipa = ["ʔ", "b", "t", "θ", "d͡ʒ", "dʒ", "ʒ", "ħ", "x", "d", "ð", "r", "z", "s", "ʃ", "sˤ", "dˤ", "tˤ", "ðˤ", "ʕ", "ɣ", "f", "q", "k", "l", "m", "n", "h", "w", "j", "lˤ", "rˤ"]
_vowels_and_modifiers = ["a", "i", "u", "e", "o", "æ", "ɑ", "ɒ", "ɔ", "ə", "ɛ", "ɜ", "ɪ", "ʊ", "ʌ", "ʏ", "ø", "aː", "iː", "uː", "eː", "oː", "ɔː", "ɑː", "ũ", "ã", "ĩ", "ː", "̃", "ˤ", "ˈ", "ˌ", "’", "ʼ", "."]
_other_consonants = ["p", "v", "g", "ŋ", "tʃ", "t͡ʃ", "ts", "dz", "ç", "ɲ", "ɾ", "ɹ", "ɬ", "ɮ"]
symbols = []
seen = set()
for s in ([_pad, _blank, _unk, _bos, _eos, _space, _word_boundary] + _silence + _punctuation + _arabic_ipa + _vowels_and_modifiers + _other_consonants):
    if s not in seen:
        symbols.append(s)
        seen.add(s)
'''
for sym_name in ("symbols.py", "symbols_exp.py"):
    for base_p in (LOCAL_REPO, os.path.join(LOCAL_REPO, "grainspeech")):
        target_sym = os.path.join(base_p, "text", sym_name)
        if os.path.exists(os.path.dirname(target_sym)):
            with open(target_sym, "w", encoding="utf-8") as f:
                f.write(repo_symbols_patch.strip() + "\n")

for ti_f in (os.path.join(LOCAL_REPO, "grainspeech", "text", "__init__.py"), os.path.join(LOCAL_REPO, "text", "__init__.py")):
    if os.path.exists(ti_f):
        with open(ti_f, "r", encoding="utf-8") as f:
            ti_txt = f.read()
        if "unk_id = _symbol_to_id.get" not in ti_txt:
            patch_str = '''
def text_to_sequence(text, cleaner_names):
    text = text.strip()
    if text.startswith("{") and text.endswith("}"):
        raw_tokens = text[1:-1].split()
    elif "{" in text and "}" in text:
        m = re.search(r"\\{(.+?)\\}", text)
        raw_tokens = m.group(1).split() if m else text.split()
    else:
        raw_tokens = text.split()
    unk_id = _symbol_to_id.get("<unk>", 2)
    seq = []
    for t in raw_tokens:
        clean_t = t.strip()
        if not clean_t:
            continue
        if clean_t in _symbol_to_id:
            seq.append(_symbol_to_id[clean_t])
        elif "@" + clean_t in _symbol_to_id:
            seq.append(_symbol_to_id["@" + clean_t])
        elif clean_t.startswith("@") and clean_t[1:] in _symbol_to_id:
            seq.append(_symbol_to_id[clean_t[1:]])
        else:
            seq.append(unk_id)
    return seq
'''
            with open(ti_f, "a", encoding="utf-8") as f:
                f.write(patch_str)

getitem_target = '        x = {"phoneme": phoneme,'
getitem_patch = '''        _min_len = min(len(phoneme), len(pitch), len(energy), len(duration))
        if _min_len > 0:
            phoneme = phoneme[:_min_len]
            pitch = pitch[:_min_len]
            energy = energy[:_min_len]
            duration = duration[:_min_len]
        x = {"phoneme": phoneme,'''

collate_target = '''        phoneme_lens = np.array([phoneme.shape[0] for phoneme in phonemes])
        mel_lens = np.array([mel.shape[0] for mel in mels])

        phonemes = pad_1D(phonemes)'''
collate_patch = '''        _collate_harmonized = True
        for _ci in range(len(phonemes)):
            _ml = min(len(phonemes[_ci]), len(pitches[_ci]), len(energies[_ci]), len(durations[_ci]))
            phonemes[_ci] = phonemes[_ci][:_ml]
            pitches[_ci] = pitches[_ci][:_ml]
            energies[_ci] = energies[_ci][:_ml]
            durations[_ci] = durations[_ci][:_ml]
        phoneme_lens = np.array([phoneme.shape[0] for phoneme in phonemes])
        mel_lens = np.array([mel.shape[0] for mel in mels])
        phonemes = pad_1D(phonemes)'''

for dm_f in (os.path.join(LOCAL_REPO, "grainspeech", "datamodule.py"), os.path.join(LOCAL_REPO, "datamodule.py")):
    if os.path.exists(dm_f):
        with open(dm_f, "r", encoding="utf-8") as f:
            dm_txt = f.read()
        if "_min_len" not in dm_txt and getitem_target in dm_txt:
            dm_txt = dm_txt.replace(getitem_target, getitem_patch)
        if "pin_memory=True" in dm_txt:
            dm_txt = dm_txt.replace(", pin_memory=True", "")
        with open(dm_f, "w", encoding="utf-8") as f:
            f.write(dm_txt)

for ml_f in (os.path.join(LOCAL_REPO, "grainspeech", "model_l1_ssim_gvar.py"), os.path.join(LOCAL_REPO, "model_l1_ssim_gvar.py")):
    if os.path.exists(ml_f):
        with open(ml_f, "r", encoding="utf-8") as f:
            ml_txt = f.read()
        pat = r'losses\s*=\s*\{"loss":\s*loss.*?self\.training_step_outputs\.clear\(\)'
        new_block = '''self.log("mel", mel_loss.detach(), on_step=False, on_epoch=True, prog_bar=True, sync_dist=True)
        self.log("l1", mel_l1_loss.detach(), on_step=False, on_epoch=True, prog_bar=True, sync_dist=True)
        self.log("ssim", ssim_loss.detach(), on_step=False, on_epoch=True, prog_bar=True, sync_dist=True)
        self.log("gvar", gvar_loss.detach(), on_step=False, on_epoch=True, prog_bar=True, sync_dist=True)
        self.log("pitch", pitch_loss.detach(), on_step=False, on_epoch=True, prog_bar=True, sync_dist=True)
        self.log("energy", energy_loss.detach(), on_step=False, on_epoch=True, prog_bar=True, sync_dist=True)
        self.log("dur", duration_loss.detach(), on_step=False, on_epoch=True, prog_bar=True, sync_dist=True)
        self.log("loss", loss.detach(), on_step=True, on_epoch=True, prog_bar=True, sync_dist=True)
        return loss


    def on_train_epoch_end(self):
        self.log("lr", self.scheduler.get_last_lr()[0], on_epoch=True, prog_bar=True, sync_dist=True)
        if hasattr(self, "training_step_outputs"):
            self.training_step_outputs.clear()'''
        if re.search(pat, ml_txt, flags=re.DOTALL):
            ml_txt = re.sub(pat, new_block, ml_txt, flags=re.DOTALL)
            with open(ml_f, "w", encoding="utf-8") as f:
                f.write(ml_txt)

pad_target = '''        s = np.shape(x)[1]
        x_padded = np.pad(
            x, (0, max_len - np.shape(x)[0]), mode="constant", constant_values=PAD
        )
        return x_padded[:, :s]'''
pad_patch = '''        return np.pad(
            x, ((0, max_len - np.shape(x)[0]), (0, 0)), mode="constant", constant_values=PAD
        )'''

for tp_f in (os.path.join(LOCAL_REPO, "grainspeech", "utils", "tools.py"), os.path.join(LOCAL_REPO, "utils", "tools.py")):
    if os.path.exists(tp_f):
        with open(tp_f, "r", encoding="utf-8") as f:
            tp_txt = f.read()
        if pad_target in tp_txt:
            tp_txt = tp_txt.replace(pad_target, pad_patch)
        with open(tp_f, "w", encoding="utf-8") as f:
            f.write(tp_txt)

train_script_p = os.path.join(LOCAL_REPO, "grainspeech", "train_l1_ssim_gvar.py")
if os.path.exists(train_script_p):
    with open(train_script_p, "r", encoding="utf-8") as f:
        ts_code = f.read()
    if "weights_only" not in ts_code:
        ts_code = "import torch\nif hasattr(torch, 'load'):\n    _orig_l = torch.load\n    def _compat_l(*a, **k):\n        k['weights_only'] = False\n        return _orig_l(*a, **k)\n    torch.load = _compat_l\n" + ts_code
    if "CleanProgressCallback" not in ts_code:
        cb_code = '''
import gc
import ctypes
from lightning.pytorch.callbacks import Callback

class CleanProgressCallback(Callback):
    def __init__(self):
        super().__init__()
        try:
            self._libc = ctypes.CDLL("libc.so.6")
        except Exception:
            self._libc = None

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        del outputs, batch
        total = getattr(trainer, "num_training_batches", 0)
        cur = batch_idx + 1
        if cur % 25 == 0 or cur == total:
            gc.collect()
            if self._libc and hasattr(self._libc, "malloc_trim"):
                try:
                    self._libc.malloc_trim(0)
                except Exception:
                    pass
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            _m = trainer.callback_metrics.get("loss")
            loss_val = float(_m) if _m is not None else 0.0
            pct = (cur / total) * 100.0 if total else 0.0
            print(f"PROGRESS: Epoch {trainer.current_epoch + 1}: {pct:.1f}% | Step {cur}/{total} | Loss: {loss_val:.4f}", flush=True)

    def on_train_epoch_end(self, trainer, pl_module):
        gc.collect()
        if self._libc and hasattr(self._libc, "malloc_trim"):
            try:
                self._libc.malloc_trim(0)
            except Exception:
                pass
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        avg_loss = trainer.callback_metrics.get("loss")
        val_loss = f"{avg_loss.item():.4f}" if avg_loss is not None else ""
        print(f"EPOCH_END: Epoch {trainer.current_epoch + 1} completed | Loss: {val_loss}", flush=True)
'''
        ts_code = cb_code + "\n" + ts_code
        target_trainer = "callbacks=[checkpoint_callback],"
        repl_trainer = "enable_progress_bar=False, callbacks=[checkpoint_callback, CleanProgressCallback()],"
        ts_code = ts_code.replace(target_trainer, repl_trainer)
    with open(train_script_p, "w", encoding="utf-8") as f:
        f.write(ts_code)

latest_ckpt = get_latest_ckpt()

try:
    subprocess.run("pkill -9 -f 'train_l1_ssim_gvar' ; pkill -9 -f 'grainspeech' ; pkill -9 -f 'torch.distributed'", shell=True, check=False)
    time.sleep(1)
except Exception:
    pass

for tmp_pat in ("/tmp/*.tar*", "/tmp/*.zip", "/tmp/*.npy", "/tmp/preprocessed_data*", "/tmp/wavs*"):
    for tmp_f in glob.glob(tmp_pat):
        try:
            os.remove(tmp_f)
        except Exception:
            pass

import gc
gc.collect()

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["NCCL_P2P_DISABLE"] = "1"
os.environ["NCCL_IB_DISABLE"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["MALLOC_ARENA_MAX"] = "2"
os.environ["MALLOC_TRIM_THRESHOLD_"] = "100000"

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    if hasattr(torch.cuda, "ipc_collect"):
        torch.cuda.ipc_collect()
    device_desc = f"{torch.cuda.get_device_name(0)} (Single GPU)"
    accelerator_choice = "gpu"
    devices_count = 1
    workers_per_gpu = 0
    actual_batch_size = min(32, BATCH_SIZE)
else:
    device_desc = "CPU"
    accelerator_choice = "cpu"
    devices_count = 1
    workers_per_gpu = 0
    actual_batch_size = min(16, BATCH_SIZE)

train_cmd = [
    sys.executable, "-u", "grainspeech/train_l1_ssim_gvar.py",
    "--run-name", EXPERIMENT_NAME,
    "--preprocess-config", config_yaml_path,
    "--hifigan-checkpoint", HIFIGAN_CKPT,
    "--accelerator", accelerator_choice,
    "--devices", str(devices_count),
    "--precision", OPTIMAL_PRECISION,
    "--batch-size", str(actual_batch_size),
    "--lr", "0.001",
    "--weight-decay", "0.00001",
    "--num_workers", str(workers_per_gpu),
    "--max_epochs", "5000",
    "--infer-device", "cuda" if torch.cuda.is_available() else "cpu",
]

if latest_ckpt:
    print(f"[GrainSpeech] Resuming training from checkpoint: {latest_ckpt}")
    train_cmd.extend(["--checkpoint", latest_ckpt])
else:
    print("[GrainSpeech] Starting training from scratch (no previous checkpoint found).")

current_progress = {
    "step": 0,
    "epoch": 0,
    "loss": "",
    "pct": "0.0%"
}
last_progress_line = [""]
recent_lines = []

def format_single_last_ckpt_name(ckpt_path):
    fname = os.path.basename(ckpt_path)
    m_ep = re.search(r"epoch[=_]?(\\d+)", fname, re.IGNORECASE)
    m_st = re.search(r"step[=_]?(\\d+)", fname, re.IGNORECASE)
    if m_ep and m_st:
        ep = int(m_ep.group(1))
        st = int(m_st.group(1))
        return f"epoch_{ep}_step_{st}_last.ckpt"
    try:
        meta = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        ep = meta.get("epoch", 0)
        st = meta.get("global_step", 0)
        return f"epoch_{ep}_step_{st}_last.ckpt"
    except Exception:
        clean_name = fname.replace(".ckpt", "").replace("=", "_")
        return f"{clean_name}_last.ckpt"

def upload_and_cleanup_ckpts(src_ckpt):
    target_name = format_single_last_ckpt_name(src_ckpt)
    target_path = f"{HF_CHECKPOINTS_PREFIX}/{target_name}"

    hf_upload_file(src_ckpt, target_path)

    old_hf_ckpts = [f for f in hf_list_files(HF_CHECKPOINTS_PREFIX) if f.endswith(".ckpt")]
    to_delete = [f for f in old_hf_ckpts if f != target_path]
    if to_delete:
        hf_delete_files(to_delete)

    hf_invalidate_cache()
    print(f"[GrainSpeech] Checkpoint uploaded: {target_name}", flush=True)
    send_telegram(f"New checkpoint saved and uploaded to Hugging Face:\nFile: {target_name}\n{last_progress_line[0]}")

stop_event = threading.Event()

def periodic_backup_loop():
    while not stop_event.is_set():
        stop_event.wait(1800)
        if stop_event.is_set():
            break
        ckpts_now = find_local_ckpts()
        if ckpts_now:
            try:
                latest_c = ckpts_now[-1]
                upload_and_cleanup_ckpts(latest_c)
            except Exception as e:
                pass

backup_thread = threading.Thread(target=periodic_backup_loop, daemon=True)
backup_thread.start()

def telegram_listener_loop():
    last_update_id = [0]
    while not stop_event.is_set():
        try:
            url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/getUpdates?offset={last_update_id[0] + 1}&timeout=5"
            req = urllib.request.Request(url)
            with urllib.request.urlopen(req, timeout=10) as resp:
                data = json.loads(resp.read().decode())
                if data.get("ok"):
                    updates = data.get("result") or []
                    for item in updates:
                        last_update_id[0] = max(last_update_id[0], item["update_id"])
                        msg = item.get("message") or item.get("channel_post")
                        if not msg:
                            continue
                        chat_id = msg["chat"]["id"]
                        cache_f = os.path.join(KAGGLE_WORKING, "telegram_chat_id.txt")
                        with open(cache_f, "w") as cf:
                            cf.write(str(chat_id))
                        text = (msg.get("text") or "").strip().lower()

                        if text in ("/logs", "logs", "/log", "log", "/full_logs", "full_logs", "سجلات", "السجلات"):
                            if recent_lines:
                                logs_text = "\n".join(recent_lines[-40:])
                                send_telegram(f"GrainSpeech Detailed Logs (Last {len(recent_lines[-40:])} lines):\n\n{logs_text}")
                            else:
                                send_telegram("No logs recorded yet. Training starting...")

                        elif text in ("/save", "/upload", "save", "upload", "حفظ", "رفع"):
                            ckpts_avail = find_local_ckpts()
                            if ckpts_avail:
                                send_telegram("Uploading latest checkpoint to Hugging Face...")
                                upload_and_cleanup_ckpts(ckpts_avail[-1])
                            else:
                                send_telegram("No checkpoint available to upload yet. Waiting for first epoch.")

                        elif text in ("/status", "status", "حالة", "الوضع", "نسبة"):
                            ep_num = current_progress.get("epoch", 1)
                            step_v = current_progress.get("step", "0")
                            loss_v = current_progress.get("loss", "N/A")
                            pct_v = current_progress.get("pct", "0.0%")
                            st_msg = (
                                f"GrainSpeech Training Status:\n"
                                f"- Epoch: {ep_num}\n"
                                f"- Step: {step_v} ({pct_v})\n"
                                f"- Loss: {loss_v}\n"
                                f"- Device: {device_desc}\n"
                                f"- Checkpoints: {len(find_local_ckpts())} local"
                            )
                            send_telegram(st_msg)

                        elif text in ("/start", "/help", "help", "مساعدة", "اوامر"):
                            help_msg = (
                                f"GrainSpeech Training Monitor Bot\n\n"
                                f"Available commands:\n"
                                f"/logs - View full detailed console logs from training process\n"
                                f"/status - Current epoch, step, loss, and training status\n"
                                f"/save - Immediately upload latest checkpoint to Hugging Face\n"
                                f"/help - Show this command menu"
                            )
                            send_telegram(help_msg)
        except Exception:
            pass
        stop_event.wait(5)

listener_thread = threading.Thread(target=telegram_listener_loop, daemon=True)
listener_thread.start()

start_time_sec = time.time()

def classify_line(line):
    line_s = line.strip()
    if not line_s:
        return "DEBUG"

    for cp in (r"Traceback", r"CUDA out of memory", r"RuntimeError", r"Error", r"Exception", r"Killed", r"Aborted", r"Segmentation fault"):
        if re.search(cp, line_s, re.IGNORECASE):
            return "CRITICAL"

    if line_s.startswith("PROGRESS:") or line_s.startswith("EPOCH_END:"):
        return "PROGRESS"

    return "DEBUG"

last_announced_epoch = [-1]

def handle_clean_progress(line):
    if line.startswith("PROGRESS:"):
        print(line, flush=True)
        m = re.search(r"Epoch\s+(\d+):\s+([0-9\.]+)%\s+\|\s+Step\s+(\d+/\d+)\s+\|\s+Loss:\s+([0-9\.]+)", line)
        if m:
            current_progress["epoch"] = int(m.group(1))
            current_progress["pct"] = f"{m.group(2)}%"
            current_progress["step"] = m.group(3)
            current_progress["loss"] = m.group(4)
    elif line.startswith("EPOCH_END:"):
        m = re.search(r"Epoch\s+(\d+)\s+completed\s+\|\s+Loss:\s*([0-9\.]*)", line)
        if m:
            ep = int(m.group(1))
            loss_val = m.group(2)
            current_progress["epoch"] = ep
            current_progress["pct"] = "100%"
            current_progress["loss"] = loss_val
            msg = f"Epoch {ep} completed | Loss: {loss_val}"
            print(msg, flush=True)
            last_progress_line[0] = msg
            if ep % 5 == 0:
                send_telegram(f"Training Progress:\n{msg}")

print(f"[GrainSpeech] Training initialized on GPU: {device_desc}", flush=True)
print(f"[GrainSpeech] Model: 264.8K parameters | Batch size: {actual_batch_size} | Precision: {OPTIMAL_PRECISION}", flush=True)

start_msg = (
    f"GrainSpeech Training Session Started\n"
    f"- Architecture: 264.8K Parameters (L1 + SSIM + GVar)\n"
    f"- Device: {device_desc}\n"
    f"- Batch size: {actual_batch_size} | Precision: {OPTIMAL_PRECISION}\n"
    f"- State: {'Resuming from ' + os.path.basename(latest_ckpt) if latest_ckpt else 'Training from scratch'}"
)
send_telegram(start_msg)

proc = subprocess.Popen(
    train_cmd,
    cwd=LOCAL_REPO,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

try:
    for raw_line in proc.stdout:
        line = raw_line.rstrip()
        recent_lines.append(line)
        if len(recent_lines) > 100:
            recent_lines.pop(0)
        cat = classify_line(line)
        if cat == "CRITICAL":
            print(line, flush=True)
        elif cat == "PROGRESS":
            handle_clean_progress(line)
except Exception as e:
    print(f"Process exception: {e}", flush=True)
finally:
    proc.wait()
    stop_event.set()

if proc.returncode != 0:
    print(f"[ERROR] Process terminated with exit code {proc.returncode}. Recent output lines:", flush=True)
    for rl in recent_lines[-20:]:
        print(rl, flush=True)
    print("[ERROR] Training stopped due to error above.", flush=True)
    send_telegram(f"Alert: Training stopped unexpectedly with exit code {proc.returncode}.\nSend /logs to view detailed logs.")
else:
    print("[GrainSpeech] Training session completed successfully.", flush=True)

    final_ckpts = find_local_ckpts()
    if final_ckpts:
        print(f"[GrainSpeech] Uploading final checkpoint: {os.path.basename(final_ckpts[-1])}", flush=True)
        upload_and_cleanup_ckpts(final_ckpts[-1])

    if os.path.exists(LOCAL_LOGS):
        hf_upload_folder(LOCAL_LOGS, HF_LOGS_PREFIX)

    final_msg = (
        f"GrainSpeech Training Completed Successfully.\n"
        f"Checkpoints and TensorBoard logs uploaded to Hugging Face:\n"
        f"{HF_BACKUP_REPO}"
    )
    send_telegram(final_msg)

    print("[GrainSpeech] Cell 6 complete: Training and synchronization finished.", flush=True)


In [ ]:
import shutil
import subprocess
import glob
import sys
import os
QUANTIZE_INT8 = False
TEST_TEXT = "مرحبا بكم في تجربة نموذج جرين سبيتش لتحويل النص الى كلام عالي الجودة"

all_c = find_local_ckpts() if "find_local_ckpts" in dir() else glob.glob(os.path.join(LOCAL_CHECKPOINTS, "*.ckpt"))
checkpoint_path = all_c[-1] if all_c else None

if checkpoint_path and os.path.exists(checkpoint_path):
    print(f"Loading checkpoint for inference & export: {checkpoint_path}")
    from model_l1_ssim_gvar import GrainSpeech, get_hifigan
    from text import text_to_sequence
    import yaml
    from IPython.display import Audio, display

    with open(config_yaml_path, "r", encoding="utf-8") as f:
        preprocess_config = yaml.safe_load(f)

    device = "cuda" if torch.cuda.is_available() else "cpu"

    try:
        model = GrainSpeech.load_from_checkpoint(
            checkpoint_path,
            preprocess_config=preprocess_config,
            hifigan_checkpoint=HIFIGAN_CKPT,
            infer_device=device,
            map_location=device,
            weights_only=False,
        )
    except Exception:
        model = GrainSpeech(
            preprocess_config=preprocess_config,
            hifigan_checkpoint=HIFIGAN_CKPT,
            infer_device=device,
        )
        ckpt_data = torch.load(checkpoint_path, map_location=device, weights_only=False)
        st = ckpt_data.get("state_dict", ckpt_data)
        model.load_state_dict(st, strict=False)

    model.eval().to(device)
    vocoder = get_hifigan(checkpoint=HIFIGAN_CKPT, infer_device=device)

    cleaners = preprocess_config["preprocessing"]["text"]["text_cleaners"]
    seq = text_to_sequence(TEST_TEXT, cleaners)
    if seq:
        in_tensor = torch.tensor([seq], dtype=torch.long, device=device)
        with torch.no_grad():
            pred = model.phoneme2mel(in_tensor)
            mel = pred[1] if isinstance(pred, (list, tuple)) else (pred["mel"] if isinstance(pred, dict) else pred)
            if mel.dim() == 3:
                mel = mel.transpose(1, 2)
            if vocoder is not None:
                wav = vocoder(mel).squeeze().cpu().numpy()
                display(Audio(wav, rate=SAMPLE_RATE))

    class GrainSpeechOnnxExport(torch.nn.Module):
        def __init__(self, m):
            super().__init__()
            self.m = m.phoneme2mel

        def forward(self, phonemes):
            out = self.m(phonemes)
            mel = out[1] if isinstance(out, (list, tuple)) else (out["mel"] if isinstance(out, dict) else out)
            return mel

    onnx_file = os.path.join(LOCAL_ONNX_EXPORT, "grainspeech_kawthar.onnx")
    wrapper = GrainSpeechOnnxExport(model)
    dummy_input = torch.randint(1, 80, (1, 30), dtype=torch.long, device=device)

    try:
        torch.onnx.export(
            wrapper,
            (dummy_input,),
            onnx_file,
            input_names=["phonemes"],
            output_names=["mel"],
            dynamic_axes={"phonemes": {1: "seq_len"}, "mel": {1: "time_frames"}},
            opset_version=14,
        )
        old_onnx = hf_list_files(HF_ONNX_PREFIX)
        if old_onnx:
            hf_delete_files(old_onnx)
        hf_upload_folder(LOCAL_ONNX_EXPORT, HF_ONNX_PREFIX)
        print("Cell 8 Complete: ONNX models exported & uploaded to Hugging Face successfully.")
    except Exception as e:
        print(f"ONNX export notice: {e}")
else:
    print("No checkpoint found for ONNX export.")
